# Síntese de Imagens — Computação Gráfica

## O que é?

**Síntese de Imagens** (ou Computação Gráfica) é a área que se ocupa de **gerar imagens a partir de descrições matemáticas** de cenas, objetos e fontes de luz. O fluxo é sempre: **Modelo 3D → Algoritmo de Renderização → Imagem 2D**.

## Características Principais

- **Entrada:** descrição geométrica (vértices, faces, materiais, luzes)
- **Saída:** imagem rasterizada ou traçada
- **Algoritmos típicos:** Rasterização (OpenGL/Vulkan), Ray Tracing, Path Tracing
- **Aplicações:** jogos, cinema de animação, CAD, simulação, realidade virtual

## Aplicação escolhida: Ray Tracing

Ray Tracing simula o caminho da luz lançando raios a partir da câmera. Para cada pixel, um raio é emitido e testado contra os objetos da cena. O modelo de iluminação de **Phong** decompõe a cor resultante em três componentes:

$$I = k_a I_a + k_d (\hat{L} \cdot \hat{N}) I_d + k_s (\hat{R} \cdot \hat{V})^n I_s$$

- $k_a$: componente ambiente (iluminação global difusa)
- $k_d (\hat{L}\cdot\hat{N})$: componente difusa (rugosidade — independe do ângulo do observador)
- $k_s (\hat{R}\cdot\hat{V})^n$: componente especular (brilho — depende do ângulo do observador)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass
from typing import Optional, Tuple

## Estruturas de dados da cena

In [ ]:
@dataclass
class Vec3:
    """Vetor 3D com operações básicas."""
    x: float
    y: float
    z: float

    def __add__(self, o): return Vec3(self.x+o.x, self.y+o.y, self.z+o.z)
    def __sub__(self, o): return Vec3(self.x-o.x, self.y-o.y, self.z-o.z)
    def __mul__(self, s): return Vec3(self.x*s, self.y*s, self.z*s)
    def __rmul__(self, s): return self.__mul__(s)
    def __neg__(self): return Vec3(-self.x, -self.y, -self.z)

    def dot(self, o) -> float:
        return self.x*o.x + self.y*o.y + self.z*o.z

    def length(self) -> float:
        return np.sqrt(self.dot(self))

    def normalize(self):
        l = self.length()
        return Vec3(self.x/l, self.y/l, self.z/l)

    def to_array(self) -> np.ndarray:
        return np.array([self.x, self.y, self.z])


@dataclass
class Sphere:
    center: Vec3
    radius: float
    color: Vec3       # RGB em [0,1]
    specular: float   # expoente de brilho Phong
    reflectivity: float = 0.0


@dataclass
class Light:
    position: Vec3
    intensity: Vec3   # cor da luz RGB

## Algoritmo de interseção raio-esfera

Um raio é definido por $\mathbf{P}(t) = \mathbf{O} + t\mathbf{D}$. Uma esfera de centro $\mathbf{C}$ e raio $r$ é descrita por $|\mathbf{P}-\mathbf{C}|^2 = r^2$. Substituindo:

$$t^2(\mathbf{D}\cdot\mathbf{D}) + 2t(\mathbf{D}\cdot(\mathbf{O}-\mathbf{C})) + (\mathbf{O}-\mathbf{C})\cdot(\mathbf{O}-\mathbf{C}) - r^2 = 0$$

Discriminante $\Delta = b^2 - 4ac$: se $\Delta \geq 0$ há interseção.

In [ ]:
def intersect_sphere(origin: Vec3, direction: Vec3, sphere: Sphere) -> Optional[float]:
    """Retorna t (distância) da primeira interseção raio-esfera, ou None."""
    oc = origin - sphere.center
    a = direction.dot(direction)
    b = 2.0 * oc.dot(direction)
    c = oc.dot(oc) - sphere.radius ** 2
    discriminant = b * b - 4 * a * c
    if discriminant < 0:
        return None
    t = (-b - np.sqrt(discriminant)) / (2.0 * a)
    return t if t > 1e-4 else None


def phong_lighting(point: Vec3, normal: Vec3, view_dir: Vec3,
                   sphere: Sphere, lights: list, spheres: list) -> Vec3:
    """Calcula a cor de um ponto usando o modelo de iluminação de Phong."""
    # Componente ambiente
    ambient = 0.1
    color = Vec3(sphere.color.x * ambient,
                 sphere.color.y * ambient,
                 sphere.color.z * ambient)

    for light in lights:
        light_dir = (light.position - point).normalize()

        # Teste de sombra: raio do ponto até a luz
        in_shadow = any(
            intersect_sphere(point, light_dir, s) is not None
            for s in spheres if s is not sphere
        )
        if in_shadow:
            continue

        # Difusa (Lei de Lambert)
        diff = max(0.0, normal.dot(light_dir))
        color = color + Vec3(
            sphere.color.x * light.intensity.x * diff * 0.7,
            sphere.color.y * light.intensity.y * diff * 0.7,
            sphere.color.z * light.intensity.z * diff * 0.7,
        )

        # Especular (Phong)
        reflect = (normal * (2.0 * normal.dot(light_dir)) - light_dir).normalize()
        spec = max(0.0, reflect.dot(view_dir)) ** sphere.specular
        color = color + Vec3(
            light.intensity.x * spec * 0.3,
            light.intensity.y * spec * 0.3,
            light.intensity.z * spec * 0.3,
        )

    return Vec3(
        min(1.0, color.x),
        min(1.0, color.y),
        min(1.0, color.z),
    )

## Motor de Ray Tracing

In [ ]:
def trace_ray(origin: Vec3, direction: Vec3, spheres: list, lights: list,
              depth: int = 0) -> Vec3:
    """Traça um raio e retorna a cor resultante (recursivo para reflexão)."""
    closest_t = float('inf')
    hit_sphere = None

    for sphere in spheres:
        t = intersect_sphere(origin, direction, sphere)
        if t is not None and t < closest_t:
            closest_t = t
            hit_sphere = sphere

    if hit_sphere is None:
        # Cor do céu: gradiente azul
        t = 0.5 * (direction.normalize().y + 1.0)
        return Vec3(1-t, 1-t, 1.0) * (1-t) + Vec3(0.5, 0.7, 1.0) * t

    hit_point = origin + direction * closest_t
    normal = (hit_point - hit_sphere.center).normalize()
    view_dir = (-direction).normalize()

    color = phong_lighting(hit_point, normal, view_dir, hit_sphere, lights, spheres)

    # Reflexão (até profundidade 2)
    if hit_sphere.reflectivity > 0 and depth < 2:
        reflect_dir = direction - normal * (2.0 * direction.dot(normal))
        reflect_color = trace_ray(hit_point, reflect_dir, spheres, lights, depth+1)
        r = hit_sphere.reflectivity
        color = Vec3(
            color.x * (1-r) + reflect_color.x * r,
            color.y * (1-r) + reflect_color.y * r,
            color.z * (1-r) + reflect_color.z * r,
        )

    return color


def render(width: int, height: int, spheres: list, lights: list) -> np.ndarray:
    """Renderiza a cena via Ray Tracing e retorna array HxWx3."""
    image = np.zeros((height, width, 3))
    aspect_ratio = width / height
    fov = np.pi / 3  # 60°

    camera = Vec3(0, 1, 3)

    for j in range(height):
        for i in range(width):
            # Normaliza coordenadas do pixel para [-1, 1]
            px = (2 * (i + 0.5) / width - 1) * np.tan(fov/2) * aspect_ratio
            py = (1 - 2 * (j + 0.5) / height) * np.tan(fov/2)
            direction = Vec3(px, py, -1).normalize()

            color = trace_ray(camera, direction, spheres, lights)
            image[j, i] = [color.x, color.y, color.z]

    return np.clip(image, 0, 1)

## Definição da Cena e Renderização

Montamos uma cena com 4 esferas (vermelho, verde, azul e um plano-chão espelhado) e 2 fontes de luz.

In [ ]:
spheres = [
    Sphere(center=Vec3(-1.2, 0, -3), radius=0.8, color=Vec3(0.9, 0.2, 0.2), specular=50,  reflectivity=0.1),
    Sphere(center=Vec3( 0.0, 0, -4), radius=0.8, color=Vec3(0.2, 0.8, 0.2), specular=100, reflectivity=0.3),
    Sphere(center=Vec3( 1.5, 0, -3), radius=0.8, color=Vec3(0.2, 0.2, 0.9), specular=200, reflectivity=0.05),
    # Plano como esfera gigante (chão)
    Sphere(center=Vec3(0, -1001, -3), radius=1000, color=Vec3(0.8, 0.8, 0.8), specular=10, reflectivity=0.4),
]

lights = [
    Light(position=Vec3(-3, 5, 2),  intensity=Vec3(1.0, 1.0, 1.0)),
    Light(position=Vec3( 3, 3, -1), intensity=Vec3(0.5, 0.5, 0.7)),
]

print("Renderizando... (pode levar alguns segundos)")
image = render(width=320, height=200, spheres=spheres, lights=lights)
print(f"Imagem gerada: {image.shape}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].imshow(image)
axes[0].set_title("Ray Tracing — Cena com Iluminação Phong", fontsize=13)
axes[0].axis('off')

# Diagrama explicativo do modelo Phong
ax = axes[1]
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title("Modelo de Iluminação de Phong", fontsize=13)

# Superfície
ax.plot([1, 9], [3, 3], 'k-', lw=2)
ax.text(5, 2.5, 'Superfície', ha='center', fontsize=10)

# Normal
ax.annotate('', xy=(5, 7), xytext=(5, 3),
            arrowprops=dict(arrowstyle='->', color='black', lw=1.5))
ax.text(5.2, 5.5, 'N (normal)', fontsize=9, color='black')

# Raio de luz L
ax.annotate('', xy=(5, 3), xytext=(2, 7),
            arrowprops=dict(arrowstyle='->', color='orange', lw=1.5))
ax.text(2.2, 6.2, 'L (luz)', fontsize=9, color='orange')

# Vetor de reflexão R
ax.annotate('', xy=(8, 7), xytext=(5, 3),
            arrowprops=dict(arrowstyle='->', color='red', lw=1.5))
ax.text(7.0, 6.2, 'R (reflexo)', fontsize=9, color='red')

# Vetor observador V
ax.annotate('', xy=(7, 6), xytext=(5, 3),
            arrowprops=dict(arrowstyle='->', color='blue', lw=1.5))
ax.text(6.4, 4.0, 'V (câmera)', fontsize=9, color='blue')

# Fórmula
ax.text(5, 1.5,
        r'$I = k_a I_a + k_d(\hat{L}\cdot\hat{N})I_d + k_s(\hat{R}\cdot\hat{V})^n I_s$',
        ha='center', fontsize=10, bbox=dict(boxstyle='round', facecolor='lightyellow'))

plt.tight_layout()
plt.savefig('sintese_resultado.png', dpi=150, bbox_inches='tight')
plt.show()
print("Imagem salva em sintese_resultado.png")

## Análise dos Componentes Phong Isolados

Para entender melhor o modelo, vamos renderizar a esfera central com cada componente de forma isolada.

In [ ]:
def render_component(width, height, component='full'):
    """Renderiza apenas um componente da iluminação Phong."""
    img = np.zeros((height, width, 3))
    aspect = width / height
    fov = np.pi / 3
    camera = Vec3(0, 0, 2)
    sphere = Sphere(Vec3(0,0,0), 1.0, Vec3(0.2,0.6,0.9), 50)
    light = Light(Vec3(3, 5, 3), Vec3(1,1,1))

    for j in range(height):
        for i in range(width):
            px = (2*(i+0.5)/width - 1) * np.tan(fov/2) * aspect
            py = (1 - 2*(j+0.5)/height) * np.tan(fov/2)
            d = Vec3(px, py, -1).normalize()
            t = intersect_sphere(camera, d, sphere)
            if t is None:
                continue
            p = camera + d * t
            n = (p - sphere.center).normalize()
            v = (-d).normalize()
            l = (light.position - p).normalize()
            r = (n * (2.0 * n.dot(l)) - l).normalize()

            if component == 'ambient':
                c = sphere.color * 0.15
            elif component == 'diffuse':
                diff = max(0, n.dot(l))
                c = sphere.color * diff
            elif component == 'specular':
                spec = max(0, r.dot(v)) ** sphere.specular
                c = Vec3(spec, spec, spec)
            else:
                amb = sphere.color * 0.15
                diff_v = max(0, n.dot(l))
                spec_v = max(0, r.dot(v)) ** sphere.specular
                c = Vec3(
                    min(1, amb.x + sphere.color.x*diff_v*0.7 + spec_v*0.3),
                    min(1, amb.y + sphere.color.y*diff_v*0.7 + spec_v*0.3),
                    min(1, amb.z + sphere.color.z*diff_v*0.7 + spec_v*0.3),
                )
            img[j, i] = [c.x, c.y, c.z]
    return img


components = ['ambient', 'diffuse', 'specular', 'full']
labels = ['Ambiente (kₐ)', 'Difusa (kd)', 'Especular (ks)', 'Phong Completo']

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, comp, label in zip(axes, components, labels):
    img = render_component(120, 120, comp)
    ax.imshow(np.clip(img, 0, 1))
    ax.set_title(label, fontsize=11)
    ax.axis('off')

plt.suptitle('Decomposição do Modelo de Iluminação de Phong', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## Conclusão

**Síntese de Imagens / Computação Gráfica** é a área que **cria imagens** a partir de modelos matemáticos. Diferencia-se das outras áreas porque:

| Aspecto | Síntese de Imagens |
|---------|-------------------|
| **Entrada** | Modelos 3D, câmeras, luzes (dados sintéticos) |
| **Saída** | Imagem gerada computacionalmente |
| **Operação** | Simulação física da propagação da luz |
| **Exemplos** | Render de jogos, CGI de filmes, CAD |

O Ray Tracing implementado aqui demonstra os conceitos fundamentais: interseção raio-objeto, modelo de iluminação local (Phong), sombras e reflexão.